# Comprehensive Tutorial: `fit_to_oscillator`

This notebook explores the full workflow and all major usage patterns of `fit_to_oscillator` in `empylib.nklib`, including:

- Legacy fitting mode (`y_eval=None`) using measured `n` and `k`
- Custom fitting mode with `y_eval=[fun1, fun2, ...]`
- Partial bounds
- Fixed parameters (`fixed_params`)
- Multiple named models of the same type
- Weighting strategies
- Edge cases and practical tips

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from empylib.nklib import fit_to_oscillator, multi_oscillator

np.random.seed(7)
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True

## 1) API Overview

Current function signature:

```python
fit_to_oscillator(
    x,
    y_data,
    oscillator_dict,
    y_eval=None,
    bounds=None,
    weights=None,
    fixed_params=None,
    x_units='um'
)
```

Behavior:

- If `y_eval is None`, then `y_data` must be `[n_data, k_data]` (legacy mode).
- If `y_eval` is a list/tuple of callables, each callable is evaluated as `f(lam, nk)` and compared against corresponding entry in `y_data`.
- `oscillator_dict` uses named models with mandatory `'type'` key.

In [ ]:
def rmse(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    return np.sqrt(np.mean((a - b) ** 2))

def summarize_model_dict(title, d):
    print(title)
    for mname, m in d.items():
        print(f"  - {mname}: {m}")

lam = np.linspace(0.45, 1.10, 220)  # um

## 2) Legacy Mode: Fit to Measured `n` and `k`

We generate synthetic `n,k` from a known oscillator and recover it from noisy data.

In [ ]:
osc_true_legacy = {
    'drude_core': {'type': 'drude', 'epsinf': 2.2, 'wp': 5.6, 'gamma': 0.24}
}

nk_true = multi_oscillator(lam, osc_true_legacy)
n_meas = nk_true.real + 0.01 * np.random.randn(lam.size)
k_meas = nk_true.imag + 0.005 * np.random.randn(lam.size)

osc_guess_legacy = {
    'drude_core': {'type': 'drude', 'epsinf': 1.8, 'wp': 4.9, 'gamma': 0.30}
}

osc_fit_legacy, res_legacy = fit_to_oscillator(
    lam,
    [n_meas, k_meas],
    osc_guess_legacy
)

nk_fit_legacy = multi_oscillator(lam, osc_fit_legacy)

summarize_model_dict('True model:', osc_true_legacy)
summarize_model_dict('Fitted model:', osc_fit_legacy)
print('Optimization success:', res_legacy.success)
print('RMSE(n):', rmse(n_meas, nk_fit_legacy.real))
print('RMSE(k):', rmse(k_meas, nk_fit_legacy.imag))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(lam, n_meas, '.', ms=2, alpha=0.6, label='n measured')
ax[0].plot(lam, nk_fit_legacy.real, '-', lw=2, label='n fit')
ax[0].set_title('Legacy mode: n')
ax[0].set_xlabel('Wavelength [um]')
ax[0].legend()

ax[1].plot(lam, k_meas, '.', ms=2, alpha=0.6, label='k measured')
ax[1].plot(lam, nk_fit_legacy.imag, '-', lw=2, label='k fit')
ax[1].set_title('Legacy mode: k')
ax[1].set_xlabel('Wavelength [um]')
ax[1].legend()

plt.tight_layout()
plt.show()

## 3) Fixed Parameters + Partial Bounds

Here we fix `gamma` and only fit the remaining parameters, while overriding bounds only for selected parameters.

In [ ]:
osc_guess_fixed = {
    'drude_core': {'type': 'drude', 'epsinf': 1.7, 'wp': 4.5, 'gamma': 0.24}
}

bounds_partial = {
    'drude_core': {
        'epsinf': (1.0, 3.0),
        'wp': (3.0, 8.0)
    }
}

fixed_params = {'drude_core': ['gamma']}

osc_fit_fixed, res_fixed = fit_to_oscillator(
    lam,
    [n_meas, k_meas],
    osc_guess_fixed,
    bounds=bounds_partial,
    fixed_params=fixed_params
)

print('Fitted with gamma fixed:')
summarize_model_dict('Model:', osc_fit_fixed)
print('Optimization success:', res_fixed.success)

## 4) Custom Mode: Fit to Reflectance and Transmittance

Now we fit not directly to `n,k`, but to transformed observables using custom evaluator functions.

We define:
- `fun_R(lam, nk)`
- `fun_T(lam, nk)`

Then call:

```python
fit_to_oscillator(lam, [R_measured, T_measured], oscillator_guess, y_eval=[fun_R, fun_T])
```

In [ ]:
def fun_R(lam, nk):
    # Simplified normal-incidence interface reflectance
    n = np.real(nk)
    return ((n - 1.0) / (n + 1.0)) ** 2

def fun_T(lam, nk):
    # Simplified Beer-Lambert-like transmittance proxy
    k = np.clip(np.imag(nk), 0.0, None)
    d_um = 1.0
    return np.exp(-4 * np.pi * k * d_um / lam)

osc_true_rt = {
    'drude_core': {'type': 'drude', 'epsinf': 2.0, 'wp': 5.2, 'gamma': 0.20},
    'lorentz_band': {'type': 'lorentz', 'epsinf': 1.0, 'wp': 2.1, 'wn': 3.0, 'gamma': 0.30}
}

nk_true_rt = multi_oscillator(lam, osc_true_rt)
R_meas = fun_R(lam, nk_true_rt) + 0.003 * np.random.randn(lam.size)
T_meas = fun_T(lam, nk_true_rt) + 0.004 * np.random.randn(lam.size)

osc_guess_rt = {
    'drude_core': {'type': 'drude', 'epsinf': 1.7, 'wp': 4.7, 'gamma': 0.28},
    'lorentz_band': {'type': 'lorentz', 'epsinf': 1.0, 'wp': 1.7, 'wn': 2.4, 'gamma': 0.45}
}

osc_fit_rt, res_rt = fit_to_oscillator(
    lam,
    [R_meas, T_meas],
    osc_guess_rt,
    y_eval=[fun_R, fun_T]
)

nk_fit_rt = multi_oscillator(lam, osc_fit_rt)
R_fit = fun_R(lam, nk_fit_rt)
T_fit = fun_T(lam, nk_fit_rt)

print('Optimization success:', res_rt.success)
print('RMSE(R):', rmse(R_meas, R_fit))
print('RMSE(T):', rmse(T_meas, T_fit))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(lam, R_meas, '.', ms=2, alpha=0.6, label='R measured')
ax[0].plot(lam, R_fit, '-', lw=2, label='R fit')
ax[0].set_title('Custom mode: Reflectance target')
ax[0].set_xlabel('Wavelength [um]')
ax[0].legend()

ax[1].plot(lam, T_meas, '.', ms=2, alpha=0.6, label='T measured')
ax[1].plot(lam, T_fit, '-', lw=2, label='T fit')
ax[1].set_title('Custom mode: Transmittance target')
ax[1].set_xlabel('Wavelength [um]')
ax[1].legend()

plt.tight_layout()
plt.show()

## 5) Weighting Strategies

Examples:

- Scalar weight (global)
- Legacy two-block weights `(w_n, w_k)`
- Custom per-target weights `[w_R, w_T]`

In [ ]:
# Legacy two-block weighting (n block emphasized less than k block)
osc_w_legacy, _ = fit_to_oscillator(
    lam,
    [n_meas, k_meas],
    osc_guess_legacy,
    weights=(1.0, 3.0)
)
nk_w_legacy = multi_oscillator(lam, osc_w_legacy)

print('Legacy weighted RMSE(n):', rmse(n_meas, nk_w_legacy.real))
print('Legacy weighted RMSE(k):', rmse(k_meas, nk_w_legacy.imag))

# Custom per-target weighting (R emphasized more than T)
osc_w_custom, _ = fit_to_oscillator(
    lam,
    [R_meas, T_meas],
    osc_guess_rt,
    y_eval=[fun_R, fun_T],
    weights=[4.0, 1.0]
)
nk_w_custom = multi_oscillator(lam, osc_w_custom)

print('Custom weighted RMSE(R):', rmse(R_meas, fun_R(lam, nk_w_custom)))
print('Custom weighted RMSE(T):', rmse(T_meas, fun_T(lam, nk_w_custom)))

## 6) Multiple Models of the Same Type

Named models let you use repeated oscillator types naturally (e.g., two drude components).

In [ ]:
osc_true_multi = {
    'drude_A': {'type': 'drude', 'epsinf': 1.4, 'wp': 3.5, 'gamma': 0.15},
    'drude_B': {'type': 'drude', 'epsinf': 1.0, 'wp': 2.0, 'gamma': 0.35}
}

nk_multi = multi_oscillator(lam, osc_true_multi)
n_multi = nk_multi.real + 0.008 * np.random.randn(lam.size)
k_multi = nk_multi.imag + 0.004 * np.random.randn(lam.size)

osc_guess_multi = {
    'drude_A': {'type': 'drude', 'epsinf': 1.2, 'wp': 2.8, 'gamma': 0.20},
    'drude_B': {'type': 'drude', 'epsinf': 1.1, 'wp': 2.5, 'gamma': 0.30}
}

osc_fit_multi, res_multi = fit_to_oscillator(
    lam,
    [n_multi, k_multi],
    osc_guess_multi,
    fixed_params={'drude_B': ['gamma']}
)

summarize_model_dict('True two-drude model:', osc_true_multi)
summarize_model_dict('Fitted two-drude model:', osc_fit_multi)
print('Optimization success:', res_multi.success)

## 7) Edge Case: All Parameters Fixed

If all parameters are fixed, the function returns without running iterative optimization and still computes the residual cost.

In [ ]:
osc_all_fixed = {
    'drude_core': {'type': 'drude', 'epsinf': 2.2, 'wp': 5.6, 'gamma': 0.24}
}

fixed_all = {'drude_core': ['epsinf', 'wp', 'gamma']}
osc_out, res_out = fit_to_oscillator(
    lam,
    [n_meas, k_meas],
    osc_all_fixed,
    fixed_params=fixed_all
)

print('All-fixed success flag:', getattr(res_out, 'success', None))
print('All-fixed residual cost:', getattr(res_out, 'cost', None))
summarize_model_dict('Returned model:', osc_out)

## 8) Practical Recommendations

- Start from physically reasonable initial guesses.
- Use `fixed_params` to reduce parameter correlation and improve stability.
- Use partial `bounds` to constrain only sensitive parameters.
- In custom mode, make sure each `y_eval` returns a real-valued 1D array matching the corresponding measured target length.
- Begin with simple custom evaluators, then move to more realistic forward models.